> # Dimension for PRODUCTS Context

# View Schema of Tables being used

In [0]:
df1 = spark.table("databricks_medallion_pipeline.silver.crm_products")
print("Schema of CRM_Products")
df1.printSchema()

df2 = spark.table("databricks_medallion_pipeline.silver.erp_product_category")
print("Schema of ERP_Product_Category")
df2.printSchema()

# Transformation Logic

In [0]:
query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY prd.start_date, prd.product_number) AS product_key, --SURROGATE KEY
    prd.product_id,
    prd.product_number,
    prd.product_name,
    pcat.category_id,
    pcat.category,
    pcat.subcategory,
    pcat.maintenance_flag,
    prd.product_line,
    prd.start_date
FROM databricks_medallion_pipeline.silver.crm_products prd
LEFT JOIN databricks_medallion_pipeline.silver.erp_product_category pcat
    ON prd.category_id = pcat.category_id
"""

df = spark.sql(query)

In [0]:
df.printSchema()
df.limit(10).display()

#Writing the Gold Table

In [0]:
df.write.mode("overwrite").saveAsTable("databricks_medallion_pipeline.gold.dim_products")

# Verification of Gold Table

In [0]:
%sql
SELECT * FROM databricks_medallion_pipeline.gold.dim_products LIMIT 10